# SYN pilot — run CTI-SYN against recent CISA advisories + analyse results

SYN is **enabled** in production, and its trustworthiness rests on the *input-reconstruction
leakage gate* enforced at ingest: the model must see only **observations**, never the advisory's
**conclusions** (actor attribution, ATT&CK mapping). This notebook is that gate made transparent.
If those conclusions leak into the
reconstructed inputs, a model can score well just by copying — and SYN becomes a summarisation task.

This notebook is the manual gate in notebook form. It:
1. loads **real AA-series CISA advisories** from the local cache (downloaded by
   `scripts/download_cisa_advisories.py`; falls back to a live index fetch),
2. **rolls in CCCS (Canada) + NCSC (UK) + The DFIR Report** via the multi-source connector
   (with **Malpedia** enriching the actor alias graph) and **dedupes across sources** — joint
   advisories are co-sealed, so the same report arrives from several CERTs and must be collapsed
   to one copy (preferring CISA) before scoring,
3. parses each page into sections with `parse_source_advisory` and builds a SYN item
   (reconstructed inputs + deterministic claim-set label),
4. **measures leakage** — do the synthesised conclusions (actor / technique) appear verbatim in
   the inputs? — the gate's key signal,
5. runs the model and scores recall / faithfulness / calibration,
6. reports a per-advisory table + aggregates and how to read them.

This is the transparent equivalent of `run_syn_pilot` ([syn_service.py](../../src/glokta/application/cti/syn_service.py)).
Source suitability for CCCS/NCSC/ACSC is reviewed in
[08_source_review_national_certs.ipynb](08_source_review_national_certs.ipynb).

> Prereq for live data: run `PYTHONPATH=src python scripts/download_cisa_advisories.py 100`
> once to populate `data/cisa_advisories/`. CCCS/NCSC are fetched live and best-effort (a
> blocked/timed-out source just contributes fewer items).

In [ ]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

## 0. Config
Knobs (env-overridable so the notebook stays quick).

In [ ]:
MAX_ADVISORIES = int(os.environ.get("SYN_PILOT_MAX", "5"))   # per source
USE_GALAXY = os.environ.get("SYN_PILOT_GALAXY", "1") == "1"  # fetch MISP galaxy for actor claims
MASK = os.environ.get("SYN_PILOT_MASK", "1") == "1"          # hybrid leakage policy: mask + drop
# Which sources to collect. CISA comes from the on-disk cache; CCCS/NCSC/DFIR are fetched live.
SOURCES_TO_USE = [s.strip() for s in os.environ.get("SYN_PILOT_SOURCES", "cisa,cccs,ncsc,dfir").split(",") if s.strip()]
PRIORITY = ("cisa", "cccs", "ncsc", "dfir")                  # dedup keeps the highest-priority copy
MIN_INPUT_CHARS = 200                                        # drop advisories thinner than this
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"}
CACHE_DIR = os.path.join(ROOT, "data", "cisa_advisories")
# The conclusion claim types that MUST NOT appear in the reconstructed inputs:
LEAK_TYPES = {"actor", "technique"}
print("max/source:", MAX_ADVISORIES, "| sources:", SOURCES_TO_USE, "| galaxy:", USE_GALAXY, "| mask:", MASK)

## 1. Build the actor alias graph (optional)
Lets the deterministic extractor recognise actor names. Two sources merge into one index: the **MISP galaxy** (bulk) and **Malpedia** (best-effort, capped — its API is slow). Falls back to empty/galaxy-only if a feed is unavailable.

In [ ]:
import httpx

def _add_actors(actors):
    for a in actors:
        alias_index[a.canonical_name.lower()] = a.canonical_name
        for al in (a.aliases or []):
            alias_index[al.lower()] = a.canonical_name

alias_index = {}
if USE_GALAXY:
    try:
        from glokta.infrastructure.cti.connectors.galaxy import fetch_galaxy_cluster, normalise_galaxy
        with httpx.Client(timeout=60.0, headers=HEADERS) as c:
            _add_actors(normalise_galaxy(fetch_galaxy_cluster(c)))
        print("alias index after galaxy:", len(alias_index))
    except Exception as exc:
        print("galaxy unavailable -> actor claims limited:", type(exc).__name__, str(exc)[:60])
    # Malpedia enrichment (capped; the API is slow / sometimes blocks automated clients).
    try:
        from glokta.infrastructure.cti.connectors.malpedia import fetch_malpedia_actors, normalise_malpedia_actors
        with httpx.Client(timeout=20.0, headers=HEADERS, follow_redirects=True) as c:
            _add_actors(normalise_malpedia_actors(fetch_malpedia_actors(c, limit=40)))
        print("alias index after malpedia:", len(alias_index))
    except Exception as exc:
        print("malpedia unavailable -> galaxy aliases only:", type(exc).__name__, str(exc)[:60])

## 2. Dataflow — collect advisories from CISA + CCCS + NCSC + DFIR
CISA AA pages load from the disk cache (or a live index fetch). CCCS, NCSC and **The DFIR Report** are collected live via the multi-source connector: `fetch_source_index` enumerates each source (the CERTs' listings; DFIR's RSS feed), then `parse_source_advisory` isolates the body, splits by heading, and buckets sections into observations (kept) vs conclusions/chrome (dropped) using that source's drop-keywords. Live fetches are best-effort — a blocked or slow source simply contributes fewer items.

In [ ]:
import glob, json
from glokta.infrastructure.cti.connectors.report import (
    fetch_aa_advisory_index, fetch_source_index, parse_advisory_page,
    parse_source_advisory, detect_actor)
from glokta.infrastructure.cti.claim_extraction import _INPUT_SECTIONS

def load_cisa_cache(n):
    files = sorted(glob.glob(os.path.join(CACHE_DIR, "*.html")))
    idx = {}
    if os.path.exists(os.path.join(CACHE_DIR, "index.json")):
        idx = json.load(open(os.path.join(CACHE_DIR, "index.json")))
    out = []
    for f in files[:n]:
        slug = os.path.basename(f)[:-5]
        html = open(f, encoding="utf-8").read()
        adv = parse_advisory_page({"link": idx.get(slug, slug)}, html)
        adv["source_site"] = "cisa"   # tag for cross-source dedup
        out.append(adv)
    return out

def load_cisa_live(n):
    out = []
    with httpx.Client(timeout=30.0, headers=HEADERS, follow_redirects=True) as c:
        urls = fetch_aa_advisory_index(c, max_pages=(n // 10) + 1, headers=HEADERS)[:n]
        for u in urls:
            try:
                out.append(parse_source_advisory({"link": u}, c.get(u).text, "cisa"))
            except Exception as exc:
                print("  cisa fetch failed:", u, type(exc).__name__)
    return out

def collect_live(source, n):
    out = []
    try:
        with httpx.Client(timeout=20.0, headers=HEADERS, follow_redirects=True) as c:
            urls = fetch_source_index(c, source, max_pages=2, headers=HEADERS)[:n]
            print(f"  {source}: index -> {len(urls)} urls")
            for u in urls:
                try:
                    out.append(parse_source_advisory({"link": u}, c.get(u).text, source))
                except Exception as exc:
                    print(f"    {source} fetch failed:", u, type(exc).__name__)
    except Exception as exc:
        print(f"  {source}: index unavailable ->", type(exc).__name__, str(exc)[:70])
    return out

pool = []
if "cisa" in SOURCES_TO_USE:
    cisa = load_cisa_cache(MAX_ADVISORIES) or load_cisa_live(MAX_ADVISORIES)
    print(f"cisa: {len(cisa)} advisories (cache or live)")
    pool += cisa
for s in [x for x in SOURCES_TO_USE if x != "cisa"]:   # cccs / ncsc / dfir
    got = collect_live(s, MAX_ADVISORIES)
    print(f"{s}: collected {len(got)} advisories")
    pool += got

for adv in pool:
    adv["actor"] = detect_actor(adv["text"], alias_index)

print(f"\ncollected pool: {len(pool)} advisories across {len({a['source_site'] for a in pool})} source(s)")
for adv in pool:
    kept = [s for s in _INPUT_SECTIONS if (adv.get("sections") or {}).get(s)]
    print(f"  [{adv['source_site']:4}] {adv['id'][:22]:22} body={len(adv['text']):6}c  "
          f"tech={len(adv['techniques'])} cve={len(adv['cves'])} actor={adv['actor']} kept={kept}")

**Reconstruction coverage** is the first health check: `kept` should contain `Technical Details` (and often `Indicators of Compromise`). Empty kept-sections means that source's section extractor didn't match the page — a per-source finding (see notebook 08).

## 2b. Dedupe across sources
National CERTs co-seal joint advisories, so the same report arrives from CISA **and** CCCS/NCSC
under different ids. `dedupe_advisories` fingerprints each by its **CVE set** (with a normalised-
title fallback for CVE-less reports) and keeps the highest-priority copy (`cisa > cccs > ncsc`).
This stops the benchmark double-counting an item — and stops it scoring a national copy of an
advisory the model likely trained on in its CISA form.

In [ ]:
from glokta.infrastructure.cti.dedupe import dedupe_advisories

res = dedupe_advisories(pool, priority=PRIORITY)
print(f"pool {len(pool)} -> kept {len(res.kept)}  (dropped {len(res.dropped)} duplicate(s))")
if res.dropped:
    print("\ndropped as duplicates:")
    for adv, kept_id in res.dropped:
        print(f"  [{adv['source_site']:4}] {adv['id'][:28]:28} == kept {kept_id}")
else:
    print("(no cross-source duplicates in this slice — e.g. only one source was reachable)")

advisories = res.kept   # the deduped set flows into the SYN pipeline below
print("\ndeduped set by source:",
      {s: sum(1 for a in advisories if a['source_site'] == s) for s in PRIORITY})

## 3. Build SYN items — reconstruct, **mask**, drop-residue
The hybrid leakage policy (`MASK=1`):
1. `reconstruct_inputs` keeps observation sections, drops conclusion sections;
2. `mask_conclusions` removes the synthesised *labels* still embedded in the prose — ATT&CK
   technique ids and the actor name/aliases — while keeping the behavioural evidence (masking the
   *id*, not the description, is the legitimate transform that makes SYN a synthesis task);
3. **drop** any advisory that still leaks after masking, or whose masked input is too thin.

The table below shows leakage **before vs. after** masking and the keep/drop decision. If no
advisory reconstructs at all, we fall back to two labelled fixtures.

In [ ]:
from datetime import date
from glokta.infrastructure.cti.claim_extraction import reconstruct_inputs, build_claim_set

FIXTURES = [
    {  # CLEAN: actor + techniques live only in dropped (Summary / MITRE) sections
        "id": "AA-DEMO-CLEAN", "published": date(2024, 3, 1), "actor": "APT29",
        "cves": ["CVE-2024-21887"], "techniques": ["T1190", "T1133"],
        "text": "APT29 campaign against network appliances.",
        "sections": {
            "Summary": "CISA attributes this activity to APT29 (Russian SVR).",
            "Technical Details": ("The threat actor exploited an internet-facing appliance and "
                                  "established outbound C2 beaconing to 203.0.113.10 over HTTPS."),
            "Indicators of Compromise": "203.0.113.10",
            "MITRE ATT&CK Techniques": "T1190 Exploit Public-Facing Application; T1133 External Remote Services",
            "Mitigations": "Patch appliances; restrict external remote services.",
        },
    },
    {  # LEAKS: technique T1059 is stated inline in Technical Details (an observation+conclusion blur)
        "id": "AA-DEMO-LEAK", "published": date(2024, 4, 1), "actor": "APT28",
        "cves": ["CVE-2023-23397"], "techniques": ["T1059", "T1566"],
        "text": "APT28 phishing campaign.",
        "sections": {
            "Summary": "Attributed to APT28 (GRU).",
            "Technical Details": ("The actors used T1059 command execution and delivered phishing "
                                  "emails, exploiting CVE-2023-23397; beacons reached 198.51.100.5."),
            "Indicators of Compromise": "198.51.100.5",
            "MITRE ATT&CK Techniques": "T1059; T1566 Phishing",
            "Mitigations": "Apply the vendor patch for CVE-2023-23397.",
        },
    },
]

from glokta.infrastructure.cti.claim_extraction import mask_conclusions

def _leak(inputs, cs):
    concl = [c for c in cs.claims if c.type in LEAK_TYPES]
    leaked = [c for c in concl if str(c.value).lower() in inputs.lower()]
    return len(leaked), len(concl)

source = advisories if any(reconstruct_inputs(a) for a in advisories) else FIXTURES
if source is FIXTURES:
    print("USING DEMO FIXTURES (live data unavailable / not reconstructable)\n")

syn_items = []   # (advisory, final_inputs, claim_set)
dropped = 0
print(f"{'advisory':12} {'raw_leak':>9} {'masked_leak':>12}  decision")
for adv in source:
    raw = reconstruct_inputs(adv)
    if not raw:
        continue
    cs = build_claim_set(adv, judge_infer=None)  # deterministic claims (no LLM extractor in the pilot)
    rn, rd = _leak(raw, cs)
    if not MASK:
        syn_items.append((adv, raw, cs))
        print(f"{adv['id']:12} {rn:>4}/{rd:<4} {'(masking off)':>12}  kept")
        continue
    masked = mask_conclusions(raw, actor=adv.get("actor"), alias_index=alias_index)
    mn, md_ = _leak(masked, cs)
    if mn > 0 or len(masked) < MIN_INPUT_CHARS:
        dropped += 1
        why = "residual leak" if mn > 0 else "too thin"
        print(f"{adv['id']:12} {rn:>4}/{rd:<4} {mn:>5}/{md_:<6}  DROPPED ({why})")
        continue
    syn_items.append((adv, masked, cs))
    print(f"{adv['id']:12} {rn:>4}/{rd:<4} {mn:>5}/{md_:<6}  kept")

print(f"\nkept {len(syn_items)} / {len(source)} advisories (dropped {dropped})")

## 4. Confirm the kept set is leak-free
After the mask + drop policy, the kept inputs should contain **none** of their conclusion claims
(actor / technique) — CVEs and IOCs remain, as observations. Below we verify that and show a masked
snippet so you can sanity-check the inputs still read as genuine observations.

In [ ]:
for adv, inputs, cs in syn_items:
    leaked, total = _leak(inputs, cs)
    print(f"  {adv['id']:12} residual_leak={leaked}/{total}", "OK" if leaked == 0 else "<-- STILL LEAKING")

if syn_items:
    snippet = syn_items[0][1]
    print("\nmasked-input snippet (", syn_items[0][0]["id"], "):\n", snippet[:500], "...")

## 5. Tasking + scoring — run the model and score each item
The faithfulness judge: live runs use the grounding judge (here the same HF model, for the pilot); offline uses a strict substring stub. Production uses `CTI_JUDGE_MODEL` (a stronger model).

In [ ]:
from glokta.infrastructure.cti.prompts import build_prompt
from glokta.infrastructure.cti.evaluator import evaluate_item

if LIVE:
    from glokta.infrastructure.cti.judge import make_grounding_judge
    from glokta.infrastructure.cti.inference import complete
    judge = make_grounding_judge(infer=complete, model=MODEL)  # pilot: model-as-judge
else:
    judge = lambda claim, inputs: str(claim.value).lower() in inputs.lower()

records = []
for adv, inputs, cs in syn_items:
    label = {"claims": [{"type": c.type, "value": c.value, "hedge_level": c.hedge_level}
                        for c in cs.claims]}
    prompt = build_prompt("syn", inputs)
    canned = ("Assessment: likely APT29 activity exploiting CVE-2024-21887 via T1190, "
              "with C2 beaconing to 203.0.113.10. Targeting government sectors (assessed).")
    response = run_model(prompt, canned=canned, max_tokens=400)
    ctx = {"alias_index": alias_index, "inputs": inputs, "judge": judge}
    s = evaluate_item("syn", label, response, ctx)
    concl = [c for c in cs.claims if c.type in LEAK_TYPES]
    leaked = sum(1 for c in concl if str(c.value).lower() in inputs.lower())
    f = s.breakdown["faithfulness"]
    records.append({
        "advisory": adv["id"],
        "input_chars": len(inputs),
        "claims": len(cs.claims),
        "leak_rate": round(leaked / len(concl), 2) if concl else 0.0,
        "recall": round(s.breakdown["recall"], 2),
        "faithfulness": None if f is None else round(f, 2),
        "calibration": s.breakdown["calibration"],
        "primary": round(s.score, 3),
    })
    print("scored", adv["id"])

## 6. Results analysis

In [ ]:
import pandas as pd
df = pd.DataFrame(records)
if df.empty:
    print("No SYN items were scored — reconstruction produced no inputs (see section coverage in step 2).")
    print("Likely causes:")
    print("  * the recent feed items are KEV-catalog notices, not AA-series advisories, or")
    print("  * the HTML->section extractor didn't match this advisory's heading structure.")
    print("This is the gate working as intended: SYN should NOT be enabled on data it can't reconstruct.")
else:
    print(df.to_string(index=False))
    print()
    print("AGGREGATE (means):")
    print(df[["input_chars", "leak_rate", "recall", "faithfulness", "calibration", "primary"]]
          .mean(numeric_only=True).round(3).to_string())
    print()
    high_leak = df[df["leak_rate"] > 0]
    print(f"advisories with leakage: {len(high_leak)} / {len(df)}")

## 7. How to read this — the gate decision

The hybrid policy is doing the work here:
- **Masking** drives `raw_leak` (often 30–100% on real AA advisories — they inline ATT&CK ids in
  Technical Details) down to **0** on the kept set, by removing the technique-id / actor labels
  while leaving the behavioural evidence. That's the legitimate transform that makes SYN a
  *synthesis* task rather than a copy task.
- **Drop-residue** is the cost: advisories where attribution is woven unmaskably into the prose, or
  that go too thin after masking, are removed. Watch the `kept / dropped` count — if you're
  dropping most of the corpus, masking isn't enough and the section extractor needs work.
- **Low `faithfulness`** = the model asserts claims the inputs don't support (hallucination — the
  CTI-critical failure); this is what the LLM judge is for. **`calibration`** rewards hedging.

> `leak_rate == 0` is now true **by construction** on the kept set, so it's *necessary but not
> sufficient*: pair it with a human spot-check (the masked snippet above) that the inputs still read
> as genuine observations, and verify a model can't trivially recover a masked technique with no
> behavioural evidence present.

**SYN runs in production** because this gate is enforced at ingest: the kept set is leak-free, the
drop rate is acceptable, and masked inputs read naturally. The masking + drop policy lives in
`mask_conclusions` / `build_syn_item` (`mask=True`) — wired through the `cti_ingest_syn` flow via
`ingest_syn_items(..., mask=True, alias_index=...)`. Re-run this notebook to re-validate the gate
when adding a new source.

> Caveat: faithfulness here uses the model-under-test as its own judge (or an offline stub). A real
> gate run should set `CTI_JUDGE_MODEL` to a stronger, independent model.

## Production equivalent
Once SYN items are ingested (`cti_ingest_syn` / `ingest_syn_items`) and a model row exists, the same evaluation runs via the DB-backed entrypoint:

```python
from glokta.application.cti.syn_service import run_syn_pilot
run_syn_pilot(db, 'huggingface/meta-llama/Llama-3.1-8B-Instruct', limit=5)
# then inspect cti_results + each item's input_text vs label
```